## Load the summary information and preprocessed data

In [1]:
import os
import json
import pandas as pd

from pathlib import Path

from single_sample_gsea import ss_gsea


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 6 samples and 13528 columns.
First few rows of the data:


,sample_title,patient_id,tissue,diagnosis,glioblastoma subtype,brain location,g207 treatment,age (years),Sex,survival post g207 (days),...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,Patient_1_post_G207,Patient_101,brain tumor,glioblastoma,Mesenchymal,left frontal,post-G207 inoculation,52,male,113,...,8.320274,6.762154,8.169685,8.294988,6.538236,7.240946,9.108596,8.386226,7.310538,8.257274
1,Patient_3_post_G207,Patient_103,brain tumor,glioblastoma,Classical/Proneural,left frontoparietal,post-G207 inoculation,57,female,233,...,7.968605,6.508388,7.827610,7.892162,5.862397,7.058654,8.854845,8.198946,6.976693,7.716466
2,Patient_5_post_G207,Patient_105,brain tumor,glioblastoma,Mesenchymal,right frontal,post-G207 inoculation,39,female,263,...,8.099085,6.804977,8.165538,8.067695,6.414345,7.159066,8.997143,8.503342,7.394330,7.956114
3,Patient_6_post_G207,Patient_106,brain tumor,glioblastoma,Mesenchymal,right frontoparietal,post-G207 inoculation,47,female,165,...,8.903723,7.382895,8.743258,8.590933,7.334513,7.837764,9.611611,8.810894,7.658829,8.936037
4,Patient_7_post_G207,Patient_107,brain tumor,glioblastoma,Mesenchymal,right frontal,post-G207 inoculation,66,female,623,...,8.557609,7.049055,8.461490,8.463640,6.848814,7.292998,9.292928,8.454921,7.487190,8.418266


## Load gene sets

In [2]:
MSIGDB_JSON = Path("../../resources/h.all.v2026.1.Hs.json")

with MSIGDB_JSON.open(encoding="utf-8") as f:
    gene_sets = {name: data["geneSymbols"] for name, data in json.load(f).items()}

print(f"Loaded {len(gene_sets)} gene sets.")


Loaded 50 gene sets.


## Compute single sample GSEA on the preprocessed data

Select only gene columns for the computation.

In [3]:
# IMPORTANT: This is dataset-specific and is the only variable that needs to be changed for each of the dataset.
sample_id_column = 'sample_title'

gene_expression_df = df.iloc[:, summary_data['data_summary']['clinical_features']:].copy()
gene_expression_df.index = df[sample_id_column].astype(str)
gene_expression_df = gene_expression_df.T
gene_expression_df.index.name = "gene"

result = ss_gsea(gene_expression_df, gene_sets)


## Merge the results with the clinical data

The output table should have the same number of rows as before and columns should sum up to clinical data + 50 gene sets.

In [4]:
clinical_df = df.iloc[:, :summary_data["data_summary"]["clinical_features"]].copy()
clinical_df = clinical_df.set_index(sample_id_column)

output_df = pd.concat([clinical_df, result], axis=1)
output_df = output_df.reset_index(drop=False)

output_df.to_csv(f"{GEO_ID}_preprocessed_ssgsea.csv", index=False)

# sanity check (there is 50 gene sets)
assert output_df.shape[1] == summary_data["data_summary"]["clinical_features"] + 50
